# R4: LoRA fine-tune on Colab (ADR-0004)

Runs `lora-10k-s0`, `lora-30k-s0`, `lora-full-s0` with the pre-registered settings. Nothing here
reads the golden set; evaluation happens locally afterwards.

**Before running**
1. The commit to run is pushed; put its SHA below. A run refuses uncommitted changes.
2. `make_colab_bundle.py` output (`r4_train_v0.zip` and `r4_train_v0.json`) is in
   `MyDrive/pathosearch/bundle/`. The bundle is Quilt-1M data: keep it private.
3. Runtime → Change runtime type → GPU.
4. If the repository is private, add a GitHub token as the Colab secret `GITHUB_TOKEN`.

Checkpoints and adapters are written to Drive only and are never committed. If the session
disconnects, re-run the setup cells and the same training cell: it resumes from the last
finished epoch.

In [ ]:
SHA = ""  # full or short commit SHA to run (required)
REPO = "github.com/sulekara0/pathosearch.git"
DRIVE_ROOT = "/content/drive/MyDrive/pathosearch"
BUNDLE = f"{DRIVE_ROOT}/bundle/r4_train_v0.zip"
RUNS = f"{DRIVE_ROOT}/runs"
assert SHA, "set SHA to the pushed commit"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import subprocess

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
subprocess.run(["git", "clone", "-q", url, "/content/pathosearch"], check=True)
%cd /content/pathosearch
subprocess.run(["git", "checkout", "-q", SHA], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head.startswith(SHA), (head, SHA)
print("HEAD", head)

In [ ]:
!pip -q install uv
!uv sync --locked

In [ ]:
import hashlib
import json
import zipfile

meta = json.load(open(BUNDLE.replace(".zip", ".json")))
h = hashlib.sha256()
with open(BUNDLE, "rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == meta["zip_sha256"], "bundle sha256 mismatch"
zipfile.ZipFile(BUNDLE).extractall("/content/bundle")
n = len(list(__import__("pathlib").Path("/content/bundle/images").iterdir()))
assert n == meta["images"], (n, meta["images"])
print(n, "images; bundle", meta["zip_sha256"][:12], "from commit", meta["git_sha"])

## Training

One cell per run. `--resume` continues from `<run>/checkpoints/last` when it exists.

In [ ]:
def train(scale: str, seed: int = 0) -> None:
    name = f"lora-{scale}-s{seed}"
    cmd = [
        "uv",
        "run",
        "python",
        "scripts/train_lora.py",
        "--manifest",
        "eval_data/r4_train_v0",
        "--scale",
        scale,
        "--seed",
        str(seed),
        "--images-dir",
        "/content/bundle/images",
        "--captions",
        "/content/bundle/captions.jsonl",
        "--out",
        f"{RUNS}/{name}",
        "--num-workers",
        "2",
        "--resume",
    ]
    subprocess.run(cmd, check=True)

In [ ]:
train("10k")

In [ ]:
train("30k")

In [ ]:
train("full")

## Only if time is left (ADR-0004 §5)

Seeds 1 and 2 for the same three scales. If they are not run, the R4 report states that seed
variance was not measured.

In [ ]:
# for seed in (1, 2):
#     for scale in ("10k", "30k", "full"):
#         train(scale, seed)

## Summary

In [ ]:
from pathlib import Path

for run in sorted(Path(RUNS).glob("lora-*")):
    r = json.load(open(run / "run.json"))
    print(
        f"{r['name']:14s} {r['status']:9s} step {r['global_step']}/{r['total_steps']} "
        f"best epoch {r['best_epoch']} val loss {r['best_val_loss']:.4f} "
        f"{r['seconds'] / 60:.0f} min git {r['provenance']['git_sha'][:7]} dirty {r['provenance']['dirty']}"
    )

## Afterwards (locally)

Download `runs/<name>/adapter/` and `runs/<name>/run.json` into `artifacts/r4/runs/<name>/`.
Copy `run.json` to `reports/r4_train_<name>.json` and commit only that file; adapters and
checkpoints stay out of git.